# 3 - Voice design (Qwen3-TTS 1.7B **VoiceDesign**)

Invent a narrator from a written description instead of picking one of the nine presets
or cloning a real person. A third checkpoint, different again from notebooks 1 and 2.

**Why this is interesting:** the voice becomes a brand asset you own and can describe in
words, and no real person is cloned, so there is no consent trail to maintain.

### The catch

On VoiceDesign the instruction **is** the voice. There is no separate direction argument,
so a per-page direction and the voice identity share one field. Appending a direction to a
fixed description risks the voice drifting between pages - the consistency cell below is
there to check exactly that before you rely on it.

Production currently uses the preset **vivian** (notebook 1). This notebook is for
exploring whether a designed voice should replace it.

In [1]:
# ============================ DEVICE ============================
# Choose the GPU BEFORE torch is imported.
#   "1"   -> RTX 4060   (safe: leaves the 4090 to ComfyUI)
#   "0"   -> RTX 4090   (fast, but only when ComfyUI is idle)
#   ""    -> CPU only   (very slow; works with no free GPU)
#
# The production voice worker also lives on GPU 1. While it runs there is only
# ~2.3 GB free there, which is NOT enough for the 1.7B model (~4.2 GB).
# Either stop it:  cd ~/ai_voice && docker compose down
# or set GPU = "0".
import os
GPU = "1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU
DEVICE = "cuda:0" if GPU != "" else "cpu"   # after masking, the chosen card is always cuda:0
print("CUDA_VISIBLE_DEVICES =", GPU or "(none - CPU)")

CUDA_VISIBLE_DEVICES = 1


In [2]:
import time, json, subprocess
import numpy as np, torch, soundfile as sf
from IPython.display import Audio, display
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

MODELS = "/home/pavlo/lab/TTS/models"
OUT    = "/home/pavlo/lab/TTS/out"
os.makedirs(OUT, exist_ok=True)

def gpu_free():
    if DEVICE == "cpu": return "cpu"
    return subprocess.run(["nvidia-smi","--query-gpu=index,name,memory.free","--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print(gpu_free())


********
********
 
0, NVIDIA GeForce RTX 4090, 5841 MiB
1, NVIDIA GeForce RTX 4060, 7885 MiB


In [3]:
# Loudness. Production target is -16 LUFS integrated, -1 dBTP ceiling, so narration
# sits predictably under the LTX ambient audio. Set NORMALISE = False for raw output.
NORMALISE   = True
TARGET_LUFS = -16.0

def normalise(a, sr):
    if not NORMALISE: return a
    try:
        import pyloudnorm as pyln
    except ImportError:
        print("pyloudnorm not installed - skipping"); return a
    m = pyln.Meter(sr)
    y = a * (10 ** ((TARGET_LUFS - m.integrated_loudness(a)) / 20.0))
    ceiling = 10 ** (-1.0/20.0)
    pk = float(np.max(np.abs(y)))
    if pk > ceiling: y = y * (ceiling / pk)
    return y.astype(np.float32)

def play(a, sr, name=None):
    a = normalise(np.asarray(a, dtype=np.float32), sr)
    if name:
        p = os.path.join(OUT, name if name.endswith(".wav") else name + ".wav")
        sf.write(p, a, sr, subtype="PCM_16"); print("saved", p)
    display(Audio(a, rate=sr))
    return a

## Load the VoiceDesign model

In [4]:
MODEL = MODELS + "/qwen3tts-17b-vd"      # 1.7B VoiceDesign, ~4.2 GB VRAM

t0 = time.time()
model = Qwen3TTSModel.from_pretrained(
    MODEL, device_map=DEVICE,
    dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32)
print(f"loaded in {time.time()-t0:.0f}s")

loaded in 1s


## Describe a narrator

Write what the voice should be, not what happens in the story. Age, gender, warmth,
pace, and the situation all help. These six were tested; grandmother and librarian were
the most usable for bedtime stories.

In [10]:
DESIGNS = {
  "grandmother": "A warm woman in her sixties reading a bedtime story. Gentle and unhurried, "
                 "with a slight smile in the voice, as if she has told this story many times before.",
  "young_mother": "A warm woman in her thirties, bright and affectionate, close and natural, "
                  "as if reading at the bedside to her own child.",
  "playful_uncle": "A friendly man in his forties, playful and animated, light and comic, "
                   "clearly enjoying the story and the silly characters.",
  "calm_father": "A calm man in his thirties with a low, soft voice. Unhurried and reassuring, "
                 "the voice you use to settle a child to sleep.",
  "stage_teller": "A theatrical storyteller with a rich, expressive voice and clear diction, "
                  "relishing each character, like a stage actor reading to a room of children.",
  "librarian": "A bright, clear woman's voice in her forties. Articulate, encouraging and lively, "
               "like a children's librarian at story time.",
  "custom": "female voice with clear pronansiation, british accent, quite and calm, like Sherlock Holmes",
}
for k in DESIGNS: print(k)

grandmother
young_mother
playful_uncle
calm_father
stage_teller
librarian
custom


## Generate with one description

In [11]:
VOICE    = "custom"       # a key from DESIGNS, or write your own string below
INSTRUCT = DESIGNS[VOICE]
# INSTRUCT = "A soft-spoken man with a slight Irish lilt, unhurried and kind."

LANGUAGE = "english"
SEED     = 1234                # None for a different take each run

TEXT = """One sunny day at the beach, Elsa and her friend Mickey decided to build the biggest sand castle ever! They gathered buckets and shovels, their eyes sparkling with excitement.
As they finished their castle, a tiny pirate ship appeared on the horizon! Captain Crunch and First Mate Jellybean were on a quest for treasure, and they had their eyes set on Elsa's magnificent sand castle.
"Arrr! We shall take this castle!" shouted Captain Crunch as he jumped off the ship with his tiny sword raised high. Mickey and Elsa gasped in surprise!"""

if SEED is not None:
    torch.manual_seed(SEED); np.random.seed(SEED)

t0 = time.time()
wavs, sr = model.generate_voice_design(
    text=" ".join(TEXT.split()),   # ONE call per page - never split by sentence
    instruct=INSTRUCT,
    language=LANGUAGE)
a = np.asarray(wavs[0], dtype=np.float32)
gen = time.time() - t0
print(f"{len(a)/sr:.2f}s audio in {gen:.2f}s (RTF {gen/(len(a)/sr):.3f})")
print(INSTRUCT)
_ = play(a, sr, name="design_" + VOICE)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


36.88s audio in 20.88s (RTF 0.566)
female voice with clear pronansiation, british accent, quite and calm, like Sherlock Holmes
saved /home/pavlo/lab/TTS/out/design_custom.wav


## Audition every description on the same text

In [ ]:
for name, desc in DESIGNS.items():
    if SEED is not None:
        torch.manual_seed(SEED); np.random.seed(SEED)
    wavs, sr = model.generate_voice_design(text=" ".join(TEXT.split()),
                                           instruct=desc, language=LANGUAGE)
    print("---", name)
    play(np.asarray(wavs[0], dtype=np.float32), sr, name="design_" + name)

## Consistency check - the important one

Keep the description fixed, append a different per-page direction each time, and read a
whole story. Then play page 1 against the last page: **is it still the same person?**

If it holds, you get a designed narrator *and* per-page delivery - something the preset
and cloning paths cannot give you. If the voice drifts, pick one or the other.

In [ ]:
BASE_VOICE = DESIGNS["grandmother"]

STORY = [
  ("p1", "One sunny day at the beach, Elsa and her friend Mickey decided to build the biggest sand castle ever!",
         "Warm and unhurried, settling the listener in. Bright lift on the last phrase."),
  ("p2", "\"Arrr! We shall take this castle!\" shouted Captain Crunch, waving his tiny sword.",
         "Blustering and comic, never frightening. Big on the shout."),
  ("p3", "As the stars twinkled, Elsa and Mickey sat by their castle, dreaming of their next adventure.",
         "Hushed and drowsy, bedtime register. Very gentle."),
]

full = []
for pid, text, direction in STORY:
    if SEED is not None:
        torch.manual_seed(SEED); np.random.seed(SEED)
    instruct = BASE_VOICE + " For this page: " + direction
    wavs, sr = model.generate_voice_design(text=" ".join(text.split()),
                                           instruct=instruct, language=LANGUAGE)
    a = normalise(np.asarray(wavs[0], dtype=np.float32), sr)
    sf.write(f"{OUT}/design_consistency_{pid}.wav", a, sr, subtype="PCM_16")
    print("---", pid, "|", direction)
    display(Audio(a, rate=sr))
    full.append(a); full.append(np.zeros(int(sr*0.4), dtype=np.float32))

print("--- all pages back to back: does the voice stay the same person?")
display(Audio(np.concatenate(full), rate=sr))

## Free the GPU

In [9]:
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()
print(gpu_free())

0, NVIDIA GeForce RTX 4090, 5841 MiB
1, NVIDIA GeForce RTX 4060, 6967 MiB
